# JPmart - Bronze Ingestion

Reads the raw CSV files generated by `00_generate_fake_data` from the Bronze volume and lands them as Delta tables in `jpmart.bronze`.

**Bronze principle:** land the data as-is, with zero cleaning or type coercion. Every column is read as `string`, so none of the intentional dirtiness (invalid emails, mixed date formats, negative/string prices, nulls) gets silently dropped or nulled out by schema inference. All of that gets handled explicitly and visibly in `02_transform_silver`.

## Configuration

In [0]:
from pyspark.sql.functions import lit, current_timestamp, col
from delta.tables import DeltaTable

CATALOG = "jpmart"
BRONZE_SCHEMA = "bronze"
RAW_FILES_DIR = "/Volumes/jpmart/bronze/raw_files"

# (source CSV filename, target Bronze table name)
ENTITIES = [
    ("customers.csv", "customers"),
    ("products.csv", "products"),
    ("orders.csv", "orders"),
    ("order_items.csv", "order_items"),
    ("web_events.csv", "web_events"),
]

## Batch ingestion — products

`inferSchema` is left at its default (`false`), which makes Spark treat every column as `string` — intentional for Bronze, not an oversight. `_source_file` and `_ingested_at` are added for lineage.

In [0]:
def ingest_batch_to_bronze(source_filename, table_name):
    source_path = f"{RAW_FILES_DIR}/{source_filename}"
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(source_path)
        .withColumn("_source_file", lit(source_filename))
        .withColumn("_ingested_at", current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"  -> {target_table}: {df.count():,} rows")
    return df

In [0]:

ingest_batch_to_bronze("products.csv", "products")

## Shared Auto Loader helper

Both the append-only and upsert entities read their source files the same way — only what happens to each micro-batch differs (a plain append vs. a `MERGE`). `cloudFiles.inferColumnTypes` stays at its default (`false`), so columns still land as `string`, same as the batch entity above — the Bronze "no coercion" rule applies everywhere. Checkpoint and schema locations are kept as siblings of each source folder, never inside it, so Auto Loader never mistakes them for data.

In [0]:
def autoload_stream(entity_name):
    source_dir = f"{RAW_FILES_DIR}/{entity_name}"
    schema_location = f"{RAW_FILES_DIR}/_schemas/{entity_name}"
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("header", "true")
        .load(source_dir)
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingested_at", current_timestamp())
    )


def run_autoload_stream(streaming_df, entity_name, write_fn):
    checkpoint_dir = f"{RAW_FILES_DIR}/_checkpoints/{entity_name}"
    query = (
        streaming_df.writeStream
        .option("checkpointLocation", checkpoint_dir)
        .trigger(availableNow=True)
        .foreachBatch(write_fn)
        .start()
    )
    query.awaitTermination()

## Append-only ingestion — order_items, web_events

Neither entity is ever updated once written, so each micro-batch is just appended straight into the Delta table.

In [0]:
def append_batch(batch_df, batch_id, target_table):
    (
        batch_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )

for entity_name, table_name in [("order_items", "order_items"), ("web_events", "web_events")]:
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    run_autoload_stream(
        autoload_stream(entity_name),
        entity_name,
        lambda batch_df, batch_id, t=target_table: append_batch(batch_df, batch_id, t),
    )
    print(f"  -> {target_table}: {spark.table(target_table).count():,} rows")


## Upsert ingestion — customers, orders

Each micro-batch can contain brand-new rows *and* updates to existing ones mixed together (see `simulate_new_customers_batch` /`simulate_new_orders_and_items_batch` in `00_generate_fake_data`). `MERGE INTO` needs at most one source row per key in a singlestatement, or it fails outright — so the micro-batch is de-duplicated by key first. This is a mechanical requirement of upsert semantics, not a data-cleaning decision: real conflict resolution between two competing updates in the same batch still belongs in Silver. On the very first run the target table doesn't exist yet, so that batch is just written directly instead of merged.

In [0]:
def upsert_batch(batch_df, batch_id, target_table, merge_key):
    deduped = batch_df.dropDuplicates([merge_key])

    if not spark.catalog.tableExists(target_table):
        deduped.write.format("delta").saveAsTable(target_table)
        return

    delta_table = DeltaTable.forName(spark, target_table)
    (
        delta_table.alias("target")
        .merge(deduped.alias("source"), f"target.{merge_key} = source.{merge_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

UPSERT_ENTITIES = [("customers", "customers", "customer_id"), ("orders", "orders", "order_id")]

for entity_name, table_name, merge_key in UPSERT_ENTITIES:
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    run_autoload_stream(
        autoload_stream(entity_name),
        entity_name,
        lambda batch_df, batch_id, t=target_table, k=merge_key: upsert_batch(batch_df, batch_id, t, k),
    )
    print(f"  -> {target_table}: {spark.table(target_table).count():,} rows")

## Validation

Confirm every table landed with the expected row count and that all business columns came through as plain strings. To see the incremental / upsert behavior for real: go back to `00_generate_fake_data`, run `simulate_new_customers_batch()` and/or `simulate_new_orders_and_items_batch()`, then re-run the matching cells above — row counts should move by roughly the size of the new batch, and updated customers/orders should show their new values with no duplicate rows for the same key.

In [0]:
ALL_TABLES = ["products", "order_items", "web_events", "customers", "orders"]

for table_name in ALL_TABLES:
    full_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    df = spark.table(full_name)
    print(f"{full_name}: {df.count():,} rows")
    df.printSchema()

# Spot-check: confirm the dirty values survived ingestion untouched
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.products").limit(20))


# Spot-check: no duplicate customer_ids after the upsert (unlike order_items, which can legitimately have exact-duplicate rows to clean up in Silver)
customers_bronze = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
dupe_customer_ids = (
    customers_bronze.groupBy("customer_id").count().filter("count > 1")
)
print(f"Duplicate customer_ids in Bronze after upsert: {dupe_customer_ids.count()} (expected: 0)")